In [11]:
#| default_exp model_selection

## Hyperparameters tuning methods for Univariate machine learning models

In [12]:
#| export

import numpy as np
from typing import Optional

#------------------------------------------------------------------------------
# Parametric Time Series Split
#------------------------------------------------------------------------------
class SplitTimeSeries:

    def __init__(self,
                 n_splits: int,
                 test_size: int,
                 step_size: Optional[int] = None
                 ) -> None:

        """
        A time series cross-validator that generates train/test splits with a fixed test size and a configurable step size.

        Parameters
        ----------
        n_splits : int
            Number of splits to generate.
        test_size : int
            The number of samples in each test set.
        step_size : int, optional
            The number of samples to move the test set forward for each split. If None, it defaults to the test_size, meaning non-overlapping test sets.
        """
        self.test_size = test_size
        self.step_size = test_size if step_size is None else step_size
        self.n_splits = n_splits

    def split(self, X):
        n_samples = len(X)
        split_starts = []
        # Start the last test set at the last possible position
        last_test_start = n_samples - self.test_size
        # Build test starts, moving backward with fixed step_size
        current = last_test_start
        while current >= 0:
            split_starts.append(current)
            current -= self.step_size
        split_starts = split_starts[::-1]  # Reverse to start from earliest

        # Use only the last n_splits
        split_starts = split_starts[-self.n_splits:]

        for test_start in split_starts:
            test_end = test_start + self.test_size
            train_index = np.arange(0, test_start)
            test_index = np.arange(test_start, test_end)
            yield train_index, test_index


In [32]:
#| export

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")
from typing import List, Dict, Optional, Callable, Tuple, Any, Union

def optuna_tune(
    model: Any,
    df: pd.DataFrame,
    cv_split: int,
    test_size: int,
    eval_metric: Callable,
    param_space: Dict[str, Any],
    step_size: int = None,
    eval_num: int = 100,
    multivariate: bool = False,
    warm_up_steps: int = 10,
    startup_trials: int = 5,
    candidate_exog: List[str] = None,
    pareto_bounds: Union[float, Tuple[float, float]] = (0.5, 0.999),
    eval_horizons: Union[None, int] = None,
    verbose: bool = False,
    random_state: Optional[int] = 42,
) -> Tuple[Dict[str, Any], Any, Dict[str, Any], List[str]]:

    """
    Tune forecasting model hyperparameters using time series cross-validation and Optuna.
 
    Parameters
    ----------
    model : object
        Forecasting model with .fit and .forecast methods.
    df : pd.DataFrame
        Time series data (datetime index, target column, optional exogenous features).
    cv_split : int
        Number of cross-validation splits.
    test_size : int
        Number of samples in each test fold. For ml_direct_forecaster, this will be overridden to be the maximum horizon in model.H.
    eval_metric : Callable
        Metric function to minimise.
    param_space : dict
        Each value must be a callable that accepts an Optuna `trial` and returns a value.
    step_size : int, optional
        Step size between CV folds.
    eval_num : int, optional
        Number of Optuna trials. Default 100.
    multivariate : bool, optional
        If True, uses TPE (Tree-structured Parzen Estimator) sampler for multivariate hyperparameter optimization. If False, uses default Independent sampler.
    warm_up_steps : int, optional
        The number of initial CV folds to complete before the Optuna pruner begins evaluating trial performance. This "warm-up" period prevents the pruner from prematurely terminating promising trials due to high variance or noise present in the earliest (smallest) cross-validation folds. Default 10.
    startup_trials : int, optional
        The number of complete trials (hyperparameter combinations) to evaluate before enabling the pruner. This ensures the pruner has a solid baseline median to compare against. Default 5.
    candidate_exog : List[str], optional
        List of exogenous feature names to consider for feature importance-based selection. If None, no feature selection is performed.
    pareto_bounds : Union[float, Tuple[float, float]], optional
        If a float is provided, it is used as a fixed cutoff for cumulative importance (e.g., 0.8 means keep features that explain 80% of variance). If a tuple is provided, it defines the lower and upper bounds for tuning the Pareto cutoff. Default is (0.5, 0.999), meaning the cutoff will be tuned between 50% and 99.9% of cumulative importance.
    eval_horizons : Union[None, int], optional
        If an integer is provided, the evaluation metric will be calculated starting from that horizon onward (e.g., if 5 is passed, the metric will be calculated on horizons 5, 6, 7, etc.). If None, the metric will be calculated on all horizons.
    verbose : bool, optional
        Print score for every trial. Default False.
 
    Returns
    -------
    Tuple[Dict[str, Any], Any]
        Best hyperparameters and best lags (if 'lags' is in param_space).
    """

    try:
        import optuna
    except ImportError:
        raise ImportError("optuna is required for hyperparameter optimization. Install with `pip install optuna`.")
    
    if model.get_name() == "ml_direct_forecaster":
        test_size = max(model.H) 
        eval_indices = [h - 1 for h in model.H]
    else:
        test_size = test_size

    def index_target(target_array):
        if model.get_name() == "ml_direct_forecaster":
            return target_array[eval_indices]
        else:
            return target_array

    target_col = model.target_col
    mod_name = model.model.__class__.__name__ if hasattr(model, "model") else model.get_name()

    # FIXED: Added all ml_forecaster native parameters so they are not passed to base models
    _forecaster_keys = [
        "box_cox", "lags", "box_cox_biasadj", "lag_transform"
    ]
    _skip = set(_forecaster_keys).union({"pareto_cutoff"})

    def _fit_params(params: dict) -> Optional[dict]:
        base_model = getattr(model, "model", None) 
        is_lr = isinstance(base_model, LinearRegression) 

        if is_lr:
            return {}  
        return {k: v for k, v in params.items() if k not in _skip}
        
    # NEW: Helper function to consistently apply ml_forecaster specific configurations safely
    def _apply_forecaster_params(target_model, params):
        
        # FIX: Reset cached coefficients for ms_arr so it doesn't reuse 
        # mismatched pre-fitted weights when Optuna changes the feature space!
        if target_model.get_name() == "ms_arr":
            target_model.coeffs = None
            target_model.stds = None

        if "lags" in params:
            new_lags = list(range(1, params["lags"] + 1)) if isinstance(params["lags"], int) else list(params["lags"])
            # Safely assign to the correct attribute
            if hasattr(target_model, "lags"):
                target_model.lags = new_lags
            else:
                target_model.n_lag = new_lags
                
        if 'lag_transform' in params: 
            target_model.lag_transform = params['lag_transform']
        if "box_cox" in params:
            if isinstance(params["box_cox"], bool):
                target_model.box_cox = params["box_cox"]
                target_model.lamda = None # Force estimation if boolean True
            else:
                target_model.box_cox = True
                target_model.lamda = params["box_cox"]
        if "box_cox_biasadj" in params: 
            target_model.biasadj = params["box_cox_biasadj"]
    
    tscv = SplitTimeSeries(n_splits=cv_split, test_size=test_size, step_size=step_size)
    total_len = len(df) 
    first_end = total_len - cv_split * (step_size or test_size) 
    permanent_cat_vars = model.cat_variables if mod_name != "ets" and model.cat_variables is not None else []
    
    if eval_horizons is not None:
        if model.get_name() == "ml_direct_forecaster":
            raise ValueError(f"eval_horizons is only applicable for recursive models, it should be None for direct forecaster. Please set eval_horizons to None.")

        if test_size <= eval_horizons:
            raise ValueError(f"eval_horizons cannot be greater than or equal to test_size ({test_size}) for recursive models, as there would be no predictions to evaluate. Please set eval_horizons to a value less than {test_size} or set eval_horizons to None.")
        
    def objective(trial: optuna.Trial) -> float:
        params = {name: suggest_fn(trial) for name, suggest_fn in param_space.items()}
        fit_params = _fit_params(params)
    
        active_exog = []
        if candidate_exog is not None and len(candidate_exog) > 0:
            temp_ranker = model.copy()
            
            if temp_ranker.get_name() in ("arima", "ets"):
                temp_ranker.set_params(params=fit_params)
            elif hasattr(temp_ranker, "model") and hasattr(temp_ranker.model, "set_params"):
                temp_ranker.model.set_params(**fit_params)
                
            _apply_forecaster_params(temp_ranker, params)
        
            dfl = df.copy()
            
            if mod_name in ("LinearRegression", "Lasso", "Ridge", "ElasticNet"):
                num_cols = [col for col in dfl.columns if col not in permanent_cat_vars]
                scaler = StandardScaler()
                dfl[num_cols] = scaler.fit_transform(dfl[num_cols])
                temp_ranker.fit(dfl.iloc[:first_end])
                importances = np.abs(temp_ranker.direct_models[test_size].coef_).flatten() if model.get_name() == "ml_direct_forecaster" else np.abs(temp_ranker.model.coef_).flatten()
            else:
                temp_ranker.fit(dfl.iloc[:first_end])
                importances = temp_ranker.direct_models[test_size].feature_importances_ if model.get_name() == "ml_direct_forecaster" else temp_ranker.model.feature_importances_
            
            imp_dict = dict(zip(temp_ranker.X.columns, importances))

            p_threshold = trial.suggest_float("pareto_cutoff", pareto_bounds[0], pareto_bounds[1]) if isinstance(pareto_bounds, tuple) else pareto_bounds

            all_scores_sorted = np.sort(importances)[::-1]
            cumulative_imp = np.cumsum(all_scores_sorted) / (np.sum(all_scores_sorted) + 1e-8)
            cutoff_idx = min(np.searchsorted(cumulative_imp, p_threshold), len(all_scores_sorted) - 1)
            importance_cutoff = all_scores_sorted[cutoff_idx]

            candidate_scores = []
            for feat in candidate_exog:
                score = sum(v for k, v in imp_dict.items() if k == feat)
                candidate_scores.append(score)
                if score >= importance_cutoff and score > 0:
                    active_exog.append(feat)

            if not active_exog:
                active_exog = [candidate_exog[np.argmax(candidate_scores)]]

            cols_to_use = [target_col] + permanent_cat_vars + active_exog
            df_ = df[cols_to_use]
        else:
            df_ = df.copy()

        cv_scores = []
        # set forecaster parameters based on the current trial's suggested values like learning rate, depth, iterations, etc. for the specific model being tuned
        _apply_forecaster_params(model, params)
        model_ = model.copy()
        if model_.get_name() in ("arima", "ets"):
            model_.set_params(params=fit_params) # For ARIMA and ETS models, the parameters are set using the set_params method with the fit_params dictionary containing the relevant hyperparameters.
        elif (model_.get_name() == "glm") or (model_.get_name() == "ms_arr"):
            ## do not do any parameter setting for glm since the family is set at initialization and can't be changed, and the other parameters are ml_forecaster specific and will be set in the _apply_forecaster_params function
            pass
        else:
            model_.model.set_params(**fit_params) ## ML models like CatBoost, LightGBM, XGBoost, etc., require setting parameters directly on the underlying model object
        for step_idx, (train_idx, test_idx) in enumerate(tscv.split(df_)):
            train = df_.iloc[train_idx]
            test = df_.iloc[test_idx]
            
            y_true = test[target_col].values
            y_true = index_target(y_true) 
            exog_t = test.drop(columns=[target_col]) if test.shape[1] > 1 else None
            
            model_.fit(train)
            
            y_pred = model_.forecast(H=len(test), exog=exog_t)

            if eval_horizons is not None:
                y_true = y_true[eval_horizons - 1:] 
                y_pred = y_pred[eval_horizons - 1:]

            if eval_metric.__name__ in ("MASE", "SMAE", "SRMSE", "RMSSE"):
                score = eval_metric(y_true, y_pred, train[target_col])
            else:
                score = eval_metric(y_true, y_pred)
                
            cv_scores.append(score)
        
            trial.report(np.mean(cv_scores), step_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return np.mean(cv_scores)

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=startup_trials, n_warmup_steps=warm_up_steps),
        sampler=optuna.samplers.TPESampler(n_startup_trials=startup_trials, multivariate=multivariate, seed=random_state)
    )

    if verbose:
        def optuna_print_callback(study, trial):
            if trial.state == optuna.trial.TrialState.PRUNED:
                print(f"Trial {trial.number:3} | Status: PRUNED | Pruned at step: {list(trial.intermediate_values.keys())[-1] if trial.intermediate_values else 'N/A'}")
            else:
                print(f"Trial {trial.number:3} | Status: {trial.state.name:8} | Value: {trial.value:.6f} | Best trial: {study.best_trial.number:3} | Best Value: {study.best_value:.6f}")
        study.optimize(objective, n_trials=eval_num, callbacks=[optuna_print_callback])
    else:
        study.optimize(objective, n_trials=eval_num)

    best_results = study.best_params.copy()
    best_p_threshold = best_results.pop("pareto_cutoff", pareto_bounds if not isinstance(pareto_bounds, tuple) else 0.99)
    best_lags = best_results.pop("lags", None)

    # FIXED: Now securely captures ALL wrapper properties so they aren't incorrectly mapped to the base model
    other_args = {k: best_results.pop(k) for k in _forecaster_keys if k in best_results}
    best_hparams = best_results 

    best_features = []
    if candidate_exog is not None and len(candidate_exog) > 0:
        final_ranker = model.copy()
        
        if final_ranker.get_name() in ("arima", "ets"):
            final_ranker.set_params(params={k: v for k, v in best_hparams.items() if k not in _skip})
        elif hasattr(final_ranker, "model") and hasattr(final_ranker.model, "set_params"):
            final_ranker.model.set_params(**{k: v for k, v in best_hparams.items() if k not in _skip})

        # Process the wrapper configurations identical to the objective function
        _apply_forecaster_params(final_ranker, study.best_params)

        dfl_final = df.copy()
        num_cols = [col for col in dfl_final.columns if col not in permanent_cat_vars]
        
        if mod_name in ("LinearRegression", "Lasso", "Ridge", "ElasticNet"):
            scaler = StandardScaler()
            dfl_final[num_cols] = scaler.fit_transform(dfl_final[num_cols])
            final_ranker.fit(dfl_final.iloc[:first_end])
            importances = np.abs(final_ranker.direct_models[test_size].coef_).flatten() if model.get_name() == "ml_direct_forecaster" else np.abs(final_ranker.model.coef_).flatten()
        else:
            final_ranker.fit(dfl_final.iloc[:first_end])   
            importances = final_ranker.direct_models[test_size].feature_importances_ if model.get_name() == "ml_direct_forecaster" else final_ranker.model.feature_importances_
            
        imp_dict = dict(zip(final_ranker.X.columns, importances))
        
        all_scores_sorted = np.sort(importances)[::-1]
        cumulative_imp = np.cumsum(all_scores_sorted) / (np.sum(all_scores_sorted) + 1e-8)
        cutoff_idx = min(np.searchsorted(cumulative_imp, best_p_threshold), len(all_scores_sorted) - 1)
        importance_cutoff = all_scores_sorted[cutoff_idx]

        candidate_scores = []
        for feat in candidate_exog:
            score = sum(v for k, v in imp_dict.items() if k == feat)
            candidate_scores.append(score)
            if score >= importance_cutoff and score > 0:
                best_features.append(feat)
        
        if not best_features:
            best_features = [candidate_exog[np.argmax(candidate_scores)]]

    if "lag_transform" in other_args and other_args["lag_transform"] is not None:
        other_args["lag_transform"] = [tr.get_name() for tr in other_args["lag_transform"]]
    trials_data = []
    for t in study.trials:
        t_params = t.params or {}
        trials_data.append({
            "trial_number": t.number,
            "score": t.value,
            "state": t.state.name,
            "model_params": {k: v for k, v in t_params.items() if k not in _skip},
            "lags": t_params.get("lags", None),
            "other_params": {k: v for k, v in t_params.items() if k in _skip and k != "lags"}
        })
    model.trials_df = pd.DataFrame(trials_data).sort_values("score").reset_index(drop=True)
    return best_hparams, best_lags, other_args, best_features

In [ ]:
#| hide

import pandas as pd
from peshbeen.datasets import load_wales_admissions
from peshbeen.models import ml_forecaster
from peshbeen.metrics import MAE, RMSE
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown="ignore")
from lightgbm import LGBMRegressor
wales_admissions = load_wales_admissions()
wales_admissions["day_of_week"] = wales_admissions.index.dayofweek
wales_admissions["month"] = wales_admissions.index.month
# split the data into train and test sets
train = wales_admissions[:-30]
test = wales_admissions[-30:]
cat_variables = ["day_of_week", "month"]
# import linear regression from sklearn
from sklearn.linear_model import LinearRegression
ml_linear = ml_forecaster(model=LGBMRegressor(verbose=-1),
              target_col='admissions', lags = 30,
              cat_variables=cat_variables, categorical_encoder=ohe)
ml_linear.fit(train)
# ml_linear.data_prep(train)
forecasts = ml_linear.forecast(H=30, exog=test[cat_variables])

In [15]:
#| hide
lgb_param_space = {
    "learning_rate":     lambda t: t.suggest_float("learning_rate", 0.001, 0.6),
    "num_leaves":        lambda t: t.suggest_int("num_leaves", 10, 200),
    "max_depth":         lambda t: t.suggest_int("max_depth", 2, 18),
    "bagging_fraction":  lambda t: t.suggest_float("bagging_fraction", 0.5, 1.0),
    "feature_fraction":  lambda t: t.suggest_float("feature_fraction", 0.5, 1.0),
    "min_data_in_leaf":  lambda t: t.suggest_int("min_data_in_leaf", 5, 100),
    "lambda_l2":         lambda t: t.suggest_float("lambda_l2", 0.0, 10.0),
    "lambda_l1":         lambda t: t.suggest_float("lambda_l1", 0.0, 10.0),
    "min_gain_to_split": lambda t: t.suggest_float("min_gain_to_split", 0.0, 20.0),
    "max_bin":           lambda t: t.suggest_int("max_bin", 100, 350),
    "top_rate":          lambda t: t.suggest_float("top_rate", 0.05, 0.4),
    "other_rate":        lambda t: t.suggest_float("other_rate", 0.05, 0.3),
    "num_iterations":    lambda t: t.suggest_int("num_iterations", 30, 200),
    "top_k":             lambda t: t.suggest_int("top_k", 8, 30),
    # "box_cox":          lambda t: t.suggest_float("box_cox", 0.0, 4),  # Example of a float parameter for box_cox
    "lags":              lambda t: t.suggest_categorical(
                             "lags", [
                                 [1,2],[3,4],[5],7, [1, 2,3]
                             ]),
    # "box_cox":          lambda t: t.suggest_float("box_cox", 0.0, 4),
    "seed":              lambda t: t.suggest_int("seed", 0, 0)  # Fixed seed for reproducibility
}

## import catboost and define catboost parameter space
from lightgbm import LGBMRegressor
# from catboost import CatBoostRegressor
# catboost_param_space = {
#     "learning_rate":     lambda t: t.suggest_float("learning_rate", 0.001, 0.6),
#     "depth":             lambda t: t.suggest_int("depth", 2, 8),
#     "iterations":        lambda t: t.suggest_int("iterations", 30, 120),
#     "lags":              lambda t: t.suggest_categorical(
#                              "lags", [
#                                  [1,2],[3,4],[5],7, [1, 2,3]
#                              ]),
# }
ml_linear = ml_forecaster(model=LGBMRegressor(verbose=-1),
              target_col='admissions', lags = 30,
              cat_variables=cat_variables, categorical_encoder=ohe)

best_params, best_lags, other_ags, _ = optuna_tune(
    model=ml_linear,
    df=train,
    cv_split=10,
    step_size=10,
    test_size=30,
    eval_metric=RMSE,
    eval_num=5,
    warm_up_steps=2,
    param_space=lgb_param_space, verbose=True, eval_horizons=None
)

Trial   0 | Status: COMPLETE | Value: 114.329943 | Best trial:   0 | Best Value: 114.329943
Trial   1 | Status: COMPLETE | Value: 124.584289 | Best trial:   0 | Best Value: 114.329943
Trial   2 | Status: COMPLETE | Value: 146.167527 | Best trial:   0 | Best Value: 114.329943
Trial   3 | Status: COMPLETE | Value: 169.157946 | Best trial:   0 | Best Value: 114.329943
Trial   4 | Status: COMPLETE | Value: 154.702212 | Best trial:   0 | Best Value: 114.329943


In [16]:
#| hide
ml_linear = ml_forecaster(model=LGBMRegressor(verbose=-1, **best_params),
              target_col='admissions', lags = best_lags,
              cat_variables=cat_variables, categorical_encoder=ohe)
ml_linear.cross_validate(train, cv_split=10,
                         step_size=10, test_size=30,
                         metrics=[RMSE])
ml_linear.cv_summary

,eval_metric,overall_score
0,RMSE,114.329943


In [17]:
#| hide

ets_param_space = {
    "smoothing_level":     lambda t: t.suggest_float("smoothing_level", 0.001, 0.99),
    "trend":              lambda t: t.suggest_categorical(
                             "trend", [
                                 "add",
                                 "mul",
                                 None
                             ]),
    "seasonal":           lambda t: t.suggest_categorical(
                             "seasonal", [
                                 "add",
                                 "mul",
                                 None
                             ]),
    "smoothing_trend":    lambda t: t.suggest_float("smoothing_trend", 0.001, 0.99),
    "smoothing_seasonal": lambda t: t.suggest_float("smoothing_seasonal", 0.001, 0.99),
    "smoothing_level":     lambda t: t.suggest_float("smoothing_level", 0.001, 0.99),
    "seasonal_periods":              lambda t: 7,   # fixed, not sampled
}

p_values = [0, 1, 2, 3, 4]
d_values = [0, 1]
q_values = [0, 1, 2, 3, 4]
from itertools import product
orders = list(product(p_values, d_values, q_values))

P_values = [0, 1, 2, 3, 4]
D_values = [0, 1]
Q_values = [0, 1, 2, 3, 4]
seasonal_orders = list(product(P_values, D_values, Q_values))

arima_param_space = {
    "order": lambda t: t.suggest_categorical(
        "order", orders
    ),
    "seasonal_order": lambda t: t.suggest_categorical(
        "seasonal_order", seasonal_orders
    ),
    "seasonal_length": lambda t: 7,   # fixed, not sampled
}


from peshbeen.models import ets, arima
ets_model = ets(target_col='admissions')
arima_model = arima(target_col='admissions')
best_params, _, other_, _ = optuna_tune(
    model=arima_model,
    df=train,
    cv_split=10,
    step_size=10,
    test_size=30,
    eval_metric=RMSE,
    eval_num=3,
    param_space=arima_param_space, verbose=False
)

## Hyperparameter tuning for Multi-Series Interdependent Machine Learning models (ml_multi_forecaster)

In [18]:
#| export
def optuna_tune_multi(
    model: Any,
    df: pd.DataFrame,
    cv_split: int,
    test_size: int,
    eval_metric: Callable,
    param_space: Dict[str, Any],
    target_series: Optional[Union[str, List[str]]] = None,
    step_size: int = 1,
    eval_num: int = 100,
    multivariate: bool = False,
    warm_up_steps: int = 10,
    startup_trials: int = 5,
    ref_series_id: Optional[str] = None,
    transform_back: bool = True,
    verbose: bool = False,
    random_state: Optional[int] = 42
) -> Tuple[Dict[str, Any], Any, Dict[str, Any]]:
    """
    Tune multi-series panel (`ml_multi_forecaster`, `panel_forecaster`) or wide multivariate
    (`ml_multi_output_forecaster`, `var`, `ms_var`) forecasting model hyperparameters
    using time-series cross-validation and Optuna.

    Parameters
    ----------
    model : object
        An instance of `ml_multi_forecaster`, `panel_forecaster`, `ml_multi_output_forecaster`, `var`, or `ms_var`.
    df : pd.DataFrame
        Long-format panel DataFrame (with time index, id_col, target_col) or wide-format DataFrame
        (with time index and target_cols) plus optional exogenous features.
    cv_split : int
        Number of cross-validation splits.
    test_size : int
        Number of time steps in each forecast evaluation test set.
    eval_metric : Callable
        Metric function to minimize (e.g. MAE, RMSE, MAPE, MASE).
    param_space : dict
        Dictionary where each value is a callable taking an Optuna `trial`.
    target_series : str, list of str, optional
        Specific series ID(s) or target column(s) to optimize the evaluation metric for.
        - If a single string, optimizes accuracy ONLY for that series/target.
        - If a list of strings, optimizes average accuracy across those specified series/targets.
        - If None (default), optimizes average accuracy across ALL series/targets.
    step_size : int, default 1
        Step size between CV folds.
    eval_num : int, default 100
        Number of Optuna trials.
    multivariate : bool, default False
        If True, uses TPE (Tree-structured Parzen Estimator) sampler for multivariate hyperparameter optimization. If False, uses default Independent sampler.
    warm_up_steps : int, default 10
        Number of initial CV folds before trial pruning begins.
    startup_trials : int, default 5
        Number of complete trials evaluated before enabling the pruner.
    ref_series_id : str, optional
        Series ID used for date-based time index splitting in panel models (defaults to series with shortest length).
    verbose : bool, default False
        If True, prints progress for every Optuna trial.
    transform_back : bool, default True
        If True, transforms the forecasted values back to the original scale. False is recommended when there are many series with different scales, as it may lead to large errors in the evaluation metric when target scaler is passed to model.

    Returns
    -------
    Tuple[Dict[str, Any], Any, Dict[str, Any]]
        (best_hparams, best_lags, other_forecaster_args)
    """
    try:
        import optuna
    except ImportError:
        raise ImportError("optuna is required for hyperparameter optimization. Install with `pip install optuna`.")

    dfc = df.copy()
    is_wide = hasattr(model, "target_cols")

    if is_wide:
        id_col = None
        target_col = None
        series_ids = list(model.target_cols)
        ref_df = dfc
    else:
        id_col = model.id_col
        target_col = model.target_col
        series_ids = sorted(dfc[id_col].unique().tolist())
        # Determine reference series for time index splitting
        if ref_series_id is None:
            ref_series_id = min(series_ids, key=lambda s: len(dfc[dfc[id_col] == s]))
        ref_df = dfc[dfc[id_col] == ref_series_id]

    # Resolve target_series filter
    if target_series is None:
        eval_series_list = series_ids
    elif isinstance(target_series, str):
        if target_series not in series_ids:
            raise ValueError(f"Target series '{target_series}' not found in {series_ids}.")
        eval_series_list = [target_series]
    elif isinstance(target_series, list):
        for s in target_series:
            if s not in series_ids:
                raise ValueError(f"Target series '{s}' not found in {series_ids}.")
        eval_series_list = target_series
    else:
        raise TypeError("target_series must be None, str, or list of str.")

    _forecaster_keys = {
        "lags", "lag_transform", "series_encoding", "difference", "seasonal_diff",
        "trend", "pol_degree", "ets_params", "change_points", "box_cox",
        "box_cox_biasadj", "target_scaler", "cat_variables", "categorical_encoder"
    }

    def _apply_wide_forecaster_params(model_, f_params):
        t_cols = model_.target_cols
        for k, v in f_params.items():
            if k == "lags":
                if isinstance(v, int):
                    model_.n_lag = {col: list(range(1, v + 1)) for col in t_cols}
                elif isinstance(v, list):
                    model_.n_lag = {col: v for col in t_cols}
                elif isinstance(v, dict):
                    model_.n_lag = {
                        col: (list(range(1, val + 1)) if isinstance(val, int) else val)
                        for col, val in v.items()
                    }
            elif k == "lag_transform":
                model_.lag_transform = {col: v for col in t_cols} if isinstance(v, list) else v
            elif k == "difference":
                if isinstance(v, int) or v is None:
                    model_.diffs = {col: (v if v != 0 else None) for col in t_cols}
                elif isinstance(v, dict):
                    model_.diffs = {col: v.get(col, None) for col in t_cols}
            elif k == "seasonal_diff":
                if isinstance(v, int) or v is None:
                    model_.season_diffs = {col: (v if v != 0 else None) for col in t_cols}
                elif isinstance(v, dict):
                    model_.season_diffs = {col: v.get(col, None) for col in t_cols}
            elif k == "trend":
                model_.trend = {col: v for col in t_cols} if isinstance(v, str) else v
            elif k == "pol_degree":
                model_.pol = {col: v for col in t_cols} if isinstance(v, int) else {col: v.get(col, 1) for col in t_cols}
            elif k == "change_points":
                model_.cps = v
            elif k == "box_cox":
                if isinstance(v, bool):
                    model_.box_cox = {col: v for col in t_cols}
                    model_.lamdas = {col: None for col in t_cols}
                elif isinstance(v, (float, int)):
                    model_.box_cox = {col: True for col in t_cols}
                    model_.lamdas = {col: v for col in t_cols}
                elif isinstance(v, dict):
                    model_.box_cox = {}
                    model_.lamdas = {col: None for col in t_cols}
                    for col, val in v.items():
                        if isinstance(val, bool):
                            model_.box_cox[col] = val
                        else:
                            model_.box_cox[col] = True
                            model_.lamdas[col] = val
            elif k == "box_cox_biasadj":
                model_.biasadj = {col: v for col in t_cols} if isinstance(v, bool) else {col: v.get(col, False) for col in t_cols}
            else:
                setattr(model_, k, v)

    tscv = SplitTimeSeries(n_splits=cv_split, test_size=test_size, step_size=step_size)

    def objective(trial: optuna.Trial) -> float:
        params = {name: suggest_fn(trial) for name, suggest_fn in param_space.items()}
        
        forecaster_params = {k: v for k, v in params.items() if k in _forecaster_keys}
        base_model_params = {k: v for k, v in params.items() if k not in _forecaster_keys}

        model_ = model.copy()

        # Update forecaster wrapper parameters
        if is_wide:
            _apply_wide_forecaster_params(model_, forecaster_params)
        else:
            for k, v in forecaster_params.items():
                setattr(model_, k, v)

        # Update base regressor parameters
        if hasattr(model_, "model") and hasattr(model_.model, "set_params") and base_model_params:
            if model_.model.__class__.__name__ == "MultiOutputRegressor":
                wrapped_params = {}
                for k, v in base_model_params.items():
                    if k.startswith("estimator__"):
                        wrapped_params[k] = v
                    else:
                        wrapped_params[f"estimator__{k}"] = v
                model_.model.set_params(**wrapped_params)
            else:
                model_.model.set_params(**base_model_params)

        cv_scores = []

        for step_idx, (ref_train_idx, ref_test_idx) in enumerate(tscv.split(ref_df)):
            if is_wide:
                train_fold = dfc.iloc[ref_train_idx]
                test_fold = dfc.iloc[ref_test_idx]
                exog_cols = [c for c in dfc.columns if c not in series_ids]
                exog_fold = test_fold[exog_cols] if len(exog_cols) > 0 else None
                H_fold = len(ref_test_idx)
            else:
                cutoff_date = ref_df.index[ref_train_idx[-1]]
                test_dates = ref_df.index[ref_test_idx]

                train_fold = dfc[dfc.index <= cutoff_date]
                test_fold = dfc[(dfc.index > cutoff_date) & (dfc.index <= test_dates[-1])]

                exog_cols = [c for c in dfc.columns if c not in [id_col, target_col]]
                exog_fold = test_fold.drop(columns=[target_col]) if len(exog_cols) > 0 else None
                H_fold = len(test_dates)

            model_.fit(train_fold)
            fc_dict = model_.forecast(H=H_fold, exog=exog_fold)

            fold_series_scores = []
            for s in eval_series_list:
                if is_wide:
                    y_true_s = test_fold[s].values
                    y_pred_s = fc_dict[s][:len(y_true_s)]
                    y_train_s = train_fold[s].values
                else:
                    s_test_df = test_fold[test_fold[id_col] == s]
                    y_true_s = s_test_df[target_col].values
                    y_pred_s = fc_dict[s][:len(y_true_s)]
                    if not transform_back and hasattr(model_, "transform_meta"):
                        scaler = model_.transform_meta.get(s, {}).get("scaler", None)
                        if scaler is not None:
                            y_true_s = scaler.transform(y_true_s.reshape(-1, 1)).flatten()
                            y_pred_s = scaler.transform(y_pred_s.reshape(-1, 1)).flatten()
                    s_train_df = train_fold[train_fold[id_col] == s]
                    y_train_s = s_train_df[target_col].values

                if eval_metric.__name__ in ("MASE", "SMAE", "SRMSE", "RMSSE"):
                    score_s = eval_metric(y_true_s, y_pred_s, y_train_s)
                else:
                    score_s = eval_metric(y_true_s, y_pred_s)

                fold_series_scores.append(score_s)

            fold_mean = np.mean(fold_series_scores)
            cv_scores.append(fold_mean)

            trial.report(np.mean(cv_scores), step_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return np.mean(cv_scores)

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=startup_trials, n_warmup_steps=warm_up_steps),
        sampler=optuna.samplers.TPESampler(n_startup_trials=startup_trials, multivariate=multivariate, seed=random_state)
    )

    if verbose:
        def optuna_print_callback(study, trial):
            if trial.state == optuna.trial.TrialState.PRUNED:
                print(f"Trial {trial.number:3} | Status: PRUNED | Pruned at step: {list(trial.intermediate_values.keys())[-1] if trial.intermediate_values else 'N/A'}")
            else:
                print(f"Trial {trial.number:3} | Status: {trial.state.name:8} | Value: {trial.value:.6f} | Best trial: {study.best_trial.number:3} | Best Value: {study.best_value:.6f}")
        study.optimize(objective, n_trials=eval_num, callbacks=[optuna_print_callback])
    else:
        study.optimize(objective, n_trials=eval_num)

    best_results = study.best_params.copy()
    best_lags = best_results.pop("lags", None)
    other_args = {k: best_results.pop(k) for k in list(best_results.keys()) if k in _forecaster_keys}
    best_hparams = best_results

    return best_hparams, best_lags, other_args


In [19]:
from sklearn.preprocessing import OneHotEncoder
#| hide
import os
import pandas as pd
import numpy as np
from typing import Any, List, Dict, Optional, Callable, Tuple, Union
from sklearn.linear_model import Ridge
from peshbeen.metrics import MAE
from peshbeen.model_selection import SplitTimeSeries
from peshbeen.models.ml_multi_forecaster import ml_multi_forecaster
from sklearn.preprocessing import StandardScaler

# Load sales dataset using peshbeen.datasets
from peshbeen.datasets import load_sales
df_sales = load_sales()

forecaster = ml_multi_forecaster(
    model=Ridge(),
    id_col='store_item',
    target_col='sales',
    lags=7,
    id_col_encoder=OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
    target_scaler=StandardScaler()
)

param_space = {
    "alpha": lambda trial: trial.suggest_float("alpha", 1e-2, 10.0, log=True),
    # "lags": lambda trial: trial.suggest_int("lags", 3, 7),
    # "difference": lambda trial: trial.suggest_categorical("difference", [0, 1])
}

# Run test tuning across ALL series
best_hp, best_lags, other_args = optuna_tune_multi(
    model=forecaster,
    df=df_sales,
    cv_split=2,
    test_size=7,
    eval_metric=MAE,
    param_space=param_space,
    eval_num=2,
    verbose=True,
    transform_back=False
)
assert isinstance(best_hp, dict), "best_hp must be a dict"
print("Optuna Multi-Series Tuning Test Passed!")


Trial   0 | Status: COMPLETE | Value: 52301689.760848 | Best trial:   0 | Best Value: 52301689.760848
Trial   1 | Status: COMPLETE | Value: 52243283.590162 | Best trial:   1 | Best Value: 52243283.590162
Optuna Multi-Series Tuning Test Passed!


## Feature selection methods for univariate time series models

In [20]:
#| export

#------------------------------------------------------------------------------
# Feature Selection Algorithms
# ------------------------------------------------------------------------------
 
def forward_feature_selection(
    model: object,
    df: pd.DataFrame,
    cv_split: int,
    H: int,
    metric: Callable,
    step_size: Optional[int] = None,
    lags_to_consider: Optional[int] = None,
    candidate_features: Optional[List[str]] = None,
    transformations: Optional[List] = None,
    starting_lags: Optional[List[int]] = None,
    starting_transforms: Optional[List] = None,
    best_start_score: Optional[float] = None, # Changed typing to float
    improve_rate_threshold: float = 0.0,
    warm_up_steps: int = 10,
    verbose: bool = False,
):
    """
    Forward stepwise feature selection for `ml_forecaster` models.
 
    At each iteration every remaining candidate (lag, exogenous column, or
    lag-transform) is tested individually by adding it to the current best
    feature set.  The candidate that produces the largest cross-validation
    improvement is permanently added.  The loop continues until no remaining
    candidate improves any of the evaluation metrics.
 
    Parameters
    ----------
    model : object
        A *configured but unfitted* `ml_forecaster` instance.  The function works exclusively on deep copies and never mutates the object passed in.
    df : pd.DataFrame
        Full training DataFrame. Must contain the target column and any candidate exogenous columns.
    cv_split : int
        Number of time-series cross-validation folds.
    H : int
        Forecast horizon (test window size for each fold).
    step_size : int, optional
        Step size between consecutive CV folds.  If `None` (default) the step equals `H`, producing non-overlapping folds — consistent with the default behaviour of `ml_forecaster.cross_validate`.
    metric : callable or list of callable.
        One or more metric functions accepted by `ml_forecaster.cross_validate` (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a candidate is only accepted when it improves **all** metrics simultaneously.
    lags_to_consider : int, optional
        Consider lags `1, 2, ..., lags_to_consider` as candidates.  If `None`, lag selection is skipped.
    candidate_features : list of str, optional
        Column names in `df` that are exogenous feature candidates.  The function never modifies this list.  If `None`, exogenous feature selection is skipped.
    transformations : list, optional
        Lag-transform objects to test as candidates (e.g. `[rolling_mean(3, 1), expanding_std(1)]`).  The function never modifies this list.  If `None`, transform selection is skipped.
    starting_lags : list of int, optional
        Lags to include in the initial feature set before the search begins. These are *not* candidates — they are always included.  Must be a list (e.g. `[1]` or `[1, 2, 3]`).
    starting_transforms : list, optional
        Lag-transform objects to include in the initial feature set before the search begins.  Must be a list.
    best_start_score : float, optional
        Initial best score for the metric. If not provided, the function will compute the baseline score using the model with the starting features (if any) before beginning the search.
    improve_rate_threshold : float, default 0.0
        Minimum improvement rate required for a candidate to be accepted. For example, `0.05` means the new score must be at least 5% better than the current best score to be accepted.
    warm_up_steps : int, default 10
        Number of CV folds to evaluate before pruning candidates that perform worse than the current best score. This allows the function to gather enough performance data for reliable pruning decisions.
    verbose : bool, default False
        Print a message each time a candidate is accepted.
 
    Returns
    -------
    dict
        A dictionary with keys `best_lags`, `best_exogs`, and `best_transforms` containing the selected features.
    """
 
    if metric is None:
        raise ValueError("metric must be provided (a callable).")
 
    _step_size = step_size if step_size is not None else H
 
    if starting_lags is not None and not isinstance(starting_lags, list):
        raise ValueError("starting_lags must be a list of integers, e.g. [1] or [1, 2, 3].")
    if starting_transforms is not None and not isinstance(starting_transforms, list):
        raise ValueError("starting_transforms must be a list of transformation instances.")
 
    # Current best lags / transforms — kept in sync as candidates are accepted
    current_lags       = list(starting_lags) if starting_lags is not None else []
    current_transforms = list(starting_transforms) if starting_transforms is not None else []

    # Build candidate pools
    remaining_lags = (
        [x for x in range(1, lags_to_consider + 1) if x not in current_lags]
        if lags_to_consider is not None else []
    )
    remaining_feats = list(candidate_features) if candidate_features is not None else []
    remaining_transforms = list(transformations) if transformations is not None else []
 
    # Working df — start without exog candidates, add back as selected
    if candidate_features is not None:
        df_work = df.drop(columns=candidate_features)
    else:
        df_work = df.copy()
 
    best_features = {
        "best_lags":       current_lags.copy(),
        "best_exogs":      [],
        "best_transforms": current_transforms.copy(),
    }
 
    def _scores_improved(new_score, ref_score):
        """True only when every metric strictly improves."""
        if new_score == float('inf'):
            return False
        if ref_score == 0:
            return new_score < 0 # Avoid division by zero
            
        # FIXED: Added parentheses around the subtraction to ensure correct math
        improvement_rate = (ref_score - new_score) / abs(ref_score)
        return improvement_rate > improve_rate_threshold
 
    tscv = SplitTimeSeries(n_splits=cv_split, test_size=H, step_size=_step_size)

    step_score_tracker = {idx: [] for idx in range(cv_split)}

    def _validate(m, df_test, ref_score=None):
        """Fit-and-score one candidate model via manual cross-validation with early pruning."""
        cv_scores = []

        if m.get_name() == "ms_arr":
            m.coeffs = None
            m.stds = None
        
        target_col = getattr(m, "target_col", None)
        if target_col is None and hasattr(m, "models"):
            target_col = list(m.models.values())[0].target_col

        for step_idx, (train_idx, test_idx) in enumerate(tscv.split(df_test)):
            train, test = df_test.iloc[train_idx], df_test.iloc[test_idx]
            x_test = test.drop(columns=[target_col], errors="ignore")
            y_test = np.array(test[target_col])

            m.fit(train)
            exog_t = x_test if x_test.shape[1] > 0 else None
            y_pred = m.forecast(H=len(test), exog=exog_t)

            if isinstance(y_pred, pd.DataFrame):
                if m.get_name() == "pesh" and "pesh" in y_pred.columns:
                    y_pred = y_pred["pesh"].values
                elif target_col in y_pred.columns:
                    y_pred = y_pred[target_col].values
                else:
                    y_pred = y_pred.iloc[:, -1].values

            if metric.__name__ in ("MASE", "SMAE", "SRMSE", "RMSSE"):
                score = metric(y_test, y_pred, train[target_col])
            else:
                score = metric(y_test, y_pred)
            cv_scores.append(score)

            current_mean = np.mean(cv_scores)
            step_score_tracker[step_idx].append(current_mean)
            
            if ref_score is not None and ref_score != float('inf'):
                if (step_idx + 1) >= warm_up_steps:
                    if len(step_score_tracker[step_idx]) > warm_up_steps:  # Ensure we have enough data points to calculate a reliable median
                        if current_mean > np.median(step_score_tracker[step_idx][:-1]):
                            return float('inf')

        return np.mean(cv_scores)
 
    def _make_candidate_model(lags, transforms, active_exogs):
        """Return a deep copy of the base model configured with the given lags and transforms."""
        m = model.copy()
        if lags_to_consider is not None:
            # Safely assign to the correct attribute
            if hasattr(m, "lags"):
                m.lags = sorted(lags) if lags else None
            else:
                m.n_lag = sorted(lags) if lags else None
        if transformations is not None:
            m.lag_transform = list(transforms) if transforms else None
        return m
 
    # FIXED: Extract list value if user accidentally passed a list instead of float
    if best_start_score is not None:
        best_score = best_start_score[0] if isinstance(best_start_score, list) else best_start_score
    else:
        best_score = float('inf')

    # Baseline score with starting features (if any and if best_start_score isn't explicitly provided)
    if best_start_score is None and (current_lags or current_transforms or (candidate_features and len(candidate_features) > 0)):
        m_start = _make_candidate_model(current_lags, current_transforms, best_features["best_exogs"])
        best_score = _validate(m_start, df_work)
        if verbose:
            print(f"Baseline score with starting features: {best_score}")
 
    # ── Stepwise forward loop ─────────────────────────────────────────────────
    while True:
        improvement    = False
        best_candidate = {'type': None, 'value': None}
        # FIXED: Set running score to best_score at the start of loop iteration
        running_score  = best_score
 
        # ── Test lag candidates ───────────────────────────────────────────────
        if remaining_lags and lags_to_consider is not None:
            for lag in remaining_lags:
                m = _make_candidate_model(current_lags + [lag], current_transforms, best_features["best_exogs"])
                score = _validate(m, df_work, running_score)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'lag', 'value': lag}
                    improvement    = True
 
        # ── Test exogenous feature candidates ─────────────────────────────────
        if remaining_feats and candidate_features is not None:
            for feat in remaining_feats:
                # OPTIMIZED: Temporarily add the column to avoid full DataFrame copying
                df_work[feat] = df[feat]
                m = _make_candidate_model(current_lags, current_transforms, best_features["best_exogs"] + [feat])
                
                score = _validate(m, df_work, running_score)
                
                # Drop immediately after validation
                df_work.drop(columns=[feat], inplace=True)
                
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'exog', 'value': feat}
                    improvement    = True
 
        # ── Test lag-transform candidates ─────────────────────────────────────
        if remaining_transforms and transformations is not None:
            for trans in remaining_transforms:
                m = _make_candidate_model(current_lags, current_transforms + [trans], best_features["best_exogs"])
                score = _validate(m, df_work, running_score)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'transform', 'value': trans}
                    improvement    = True
 
        # ── Accept the best candidate found this round ────────────────────────
        if improvement:
            best_score = running_score
            ctype = best_candidate['type']
            cval  = best_candidate['value']
 
            if ctype == 'lag':
                current_lags.append(cval)
                current_lags.sort()
                best_features["best_lags"].append(cval)
                best_features["best_lags"].sort()
                remaining_lags.remove(cval)
 
            elif ctype == 'exog':
                best_features["best_exogs"].append(cval)
                remaining_feats.remove(cval)
                # OPTIMIZED: Permanently apply the winning column without copying
                df_work[cval] = df[cval]
 
            elif ctype == 'transform':
                current_transforms.append(cval)
                best_features["best_transforms"].append(cval)
                remaining_transforms.remove(cval)
 
            if verbose:
                label = cval.get_name() if hasattr(cval, 'get_name') else str(cval)
                print(f"Added {ctype}: {label} | score: {best_score}")
 
        else:
            break  # No candidate improved any metric — search is complete
 
    # ── Finalise output ───────────────────────────────────────────────────────
    best_features["best_transforms"] = [
        t.get_name() if hasattr(t, 'get_name') else str(t)
        for t in best_features["best_transforms"]
    ]
 
    return best_features

In [21]:
#| hide
ml_linear = ml_forecaster(model=LinearRegression(),
              target_col='admissions', lags = 30, difference=1, categorical_encoder=ohe,
              cat_variables=cat_variables)
feats = forward_feature_selection(
    df=train,
    cv_split=5,
    H=30,
    model=ml_linear,
    metric=MAE,
    lags_to_consider=15,
    transformations=None,
    starting_lags=None,
    starting_transforms=None,
    verbose=True
)

In [22]:
#| export

def backward_feature_selection(
    model: object,
    df: pd.DataFrame,
    cv_split: int,
    H: int,
    step_size: Optional[int] = None,
    metrics: Union[Callable, List[Callable]] = None,
    lags_to_consider: Optional[List[int]] = None,
    candidate_features: Optional[List[str]] = None,
    transformations: Optional[List] = None,
    verbose=False,
):
    """
    Backward stepwise feature selection for `ml_forecaster` models.

    Starts with the full feature set (all provided lags, exogenous columns, and
    lag-transforms) and at each iteration tries removing each current feature
    individually.  The feature whose removal produces the largest cross-validation
    improvement is permanently dropped.  The loop continues until no remaining
    feature can be removed without hurting any of the evaluation metrics.

    Parameters
    ----------
    model : ml_forecaster
        A *configured but unfitted* `ml_forecaster` instance.  The function works exclusively on deep copies and never mutates the object passed in.
    df : pd.DataFrame
        Full training DataFrame. Must contain the target column and any candidate exogenous columns.
    cv_split : int
        Number of time-series cross-validation folds.
    H : int
        Forecast horizon (test window size for each fold).
    step_size : int, optional
        Step size between consecutive CV folds.  If `None` (default) the step equals `H`, producing non-overlapping folds.
    metrics : callable or list of callable
        One or more metric functions accepted by `ml_forecaster.cross_validate` (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a feature is only removed when doing so improves **all** metrics simultaneously.
    lags_to_consider : list of int, optional
        Lags to include in the initial feature set and test for removal (e.g. `[1, 2, 3, 4]`).  If `None`, no lag removal is attempted.
    candidate_features : list of str, optional
        Column names in `df` that start in the model and are tested for removal.  If `None`, exogenous feature removal is skipped.
    transformations : list, optional
        Lag-transform objects that start in the model and are tested for removal (e.g. `[rolling_mean(3, 1), expanding_std(1)]`).  If `None`, transform removal is skipped.
    verbose : bool, default False
        Print a message each time a feature is removed.

    Returns
    -------
    dict
        A dictionary with keys `best_lags`, `best_exogs`, and `best_transforms` containing the surviving features after backward selection.
    """

    # ── Normalise metrics to always be a list ─────────────────────────────────
    if metrics is None:
        raise ValueError("metrics must be provided (a callable or list of callables).")
    if callable(metrics):
        metrics = [metrics]

    # ── step_size default: non-overlapping folds (step = H) ───────────────────
    _step_size = step_size if step_size is not None else H

    # ── Build initial feature sets (local copies — never mutate caller's lists) ─
    if lags_to_consider is None:
        current_lags = []
    elif isinstance(lags_to_consider, int):
        current_lags = list(range(1, lags_to_consider + 1))
    else:
        current_lags = sorted(lags_to_consider)
    current_exogs      = list(candidate_features)  if candidate_features is not None else []
    current_transforms = list(transformations)      if transformations    is not None else []


    # ── Working df includes all exog candidates from the start ────────────────
    df_work = df.copy()

    # ── Columns in candidate_features that are also cat_variables on the model ─
    _cat_exog_candidates = set()
    if candidate_features is not None and model.cat_variables is not None:
        _cat_exog_candidates = set(candidate_features) & set(model.cat_variables)

    # ── Inner helpers ─────────────────────────────────────────────────────────

    def _scores_improved(new_score, ref_score):
        """True only when every metric strictly improves."""
        return all(n < r for n, r in zip(new_score, ref_score))

    def _make_candidate_model(lags, transforms, active_exogs=None):
        """
        Return a deep copy of the base model configured with the given lags and
        transforms.  Cat_variables that are no longer in active_exogs are stripped
        from the copy so fit() does not look for missing columns.
        """
        m = model.copy()
        m.n_lag         = sorted(lags)     if lags       else None
        m.lag_transform = list(transforms) if transforms else None

        if m.cat_variables is not None and _cat_exog_candidates:
            active = set(active_exogs) if active_exogs else set()
            present_cats = [
                c for c in m.cat_variables
                if c not in _cat_exog_candidates or c in active]
            m.cat_variables = present_cats if present_cats else None

        return m

    def _validate(m, df_test):
        """Fit-and-score one candidate model via cross-validation."""
        m.cross_validate(
            df=df_test,
            cv_split=cv_split,
            test_size=H,
            metrics=metrics,
            step_size=_step_size,
        )
        result_df = m.cv_summary
        return result_df["overall_score"].tolist()

    # ── Baseline score with all features ──────────────────────────────────────
    m_start = _make_candidate_model(current_lags, current_transforms, current_exogs)
    best_score = _validate(m_start, df_work)
    if verbose:
        print(f"Baseline score with all features: {best_score}")

    # ── Stepwise backward loop ────────────────────────────────────────────────
    while True:
        improvement    = False
        best_candidate = {'type': None, 'value': None}
        running_score  = best_score

        # ── Test lag removals ─────────────────────────────────────────────────
        if current_lags:
            for lag in current_lags:
                reduced_lags = [l for l in current_lags if l != lag]
                m = _make_candidate_model(reduced_lags, current_transforms, current_exogs)
                score = _validate(m, df_work)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'lag', 'value': lag}
                    improvement    = True

        # ── Test exogenous feature removals ───────────────────────────────────
        if current_exogs:
            for feat in current_exogs:
                reduced_exogs = [f for f in current_exogs if f != feat]
                df_test = df_work.drop(columns=[feat])
                m = _make_candidate_model(current_lags, current_transforms, reduced_exogs)
                score = _validate(m, df_test)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'exog', 'value': feat}
                    improvement    = True

        # ── Test lag-transform removals ───────────────────────────────────────
        if current_transforms:
            for trans in current_transforms:
                reduced_transforms = [t for t in current_transforms if t is not trans]
                m = _make_candidate_model(current_lags, reduced_transforms, current_exogs)
                score = _validate(m, df_work)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'transform', 'value': trans}
                    improvement    = True

        # ── Accept the best removal found this round ──────────────────────────
        if improvement:
            best_score = running_score
            ctype = best_candidate['type']
            cval  = best_candidate['value']

            if ctype == 'lag':
                current_lags.remove(cval)
                if verbose:
                    print(f"Removed lag: {cval} | score: {best_score}")

            elif ctype == 'exog':
                current_exogs.remove(cval)
                df_work = df_work.drop(columns=[cval])
                if verbose:
                    print(f"Removed exog: {cval} | score: {best_score}")

            elif ctype == 'transform':
                current_transforms = [t for t in current_transforms if t is not cval]
                if verbose:
                    label = cval.get_name()
                    print(f"Removed transform: {label} | score: {best_score}")

        else:
            break  # No removal improved any metric — search is complete

    # ── Finalise output ───────────────────────────────────────────────────────
    return {
        "best_lags":       current_lags,
        "best_exogs":      current_exogs,
        "best_transforms": [t.get_name() for t in current_transforms],
    }


In [23]:
#| hide
ml_linear = ml_forecaster(model=LinearRegression(),
              target_col='admissions', lags = 30, difference=1, categorical_encoder=ohe,
              cat_variables=cat_variables)
feats = backward_feature_selection(
    df=train,
    cv_split=5,
    H=30,
    model=ml_linear,
    metrics=[MAE, RMSE],
    lags_to_consider=10,
    candidate_features=["month"],
    transformations=None,
    verbose=True
)

Baseline score with all features: [115.1433929244597, 138.64954278111077]
Removed lag: 4 | score: [111.01800509123477, 134.75574821689216]
Removed lag: 3 | score: [107.72058431919417, 131.66454938935937]
Removed lag: 7 | score: [105.55587902430713, 128.4157910758367]
Removed lag: 5 | score: [104.15345407642607, 126.84398665212048]
Removed lag: 8 | score: [103.9326299217657, 126.52895582499434]


## Feature selection methods for multivariate time series models

In [24]:
#| hide

# def mv_forward_feature_selection(
#     model: object,
#     df: pd.DataFrame,
#     target_col: str,
#     cv_split: int,
#     H: int,
#     step_size=None,
#     metrics=None,
#     lags_to_consider=None,
#     candidate_features=None,
#     transformations=None,
#     starting_lags=None,
#     starting_transforms=None,
#     verbose=False,
# ):
#     """
#     Forward stepwise feature selection for `ml_mv_forecaster`.

#     Parameters
#     ----------
#     model : ml_mv_forecaster
#         Template model — never mutated.
#     df : pd.DataFrame
#         DataFrame containing the target variable and any candidate features.
#     target_col : str
#         Target variable used to evaluate cross-validation score.
#     cv_split : int
#         Number of time-series cross-validation folds.
#     H : int
#         Forecast horizon / test size per fold.
#     step_size : int, optional
#         Rolling-window step size (defaults to H).
#     metrics : callable or list of callable
#         One or more metric functions (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a candidate is only accepted when it improves **all** metrics simultaneously.
#     lags_to_consider : dict, optional
#         ``{col: max_lag}`` — lags 1..max_lag are candidates.
#     candidate_features : list of str, optional
#         Exogenous columns to consider adding.
#     transformations : dict, optional
#         ``{col: [transform_objects]}`` — transform candidates per target.
#     starting_lags : dict, optional
#         Lags already included before search begins.
#     starting_transforms : dict, optional
#         Transforms already included before search begins.
#     verbose : bool, default False

#     Returns
#     -------
#     dict
#         `{"best_lags": {col: [...]}, "best_exogs": [...],
#            "best_transforms": {col: [name_str, ...]}}`
#     """
#     if metrics is None:
#         raise ValueError("metrics must be provided.")
#     if callable(metrics):
#         metrics = [metrics]

#     _step_size = step_size if step_size is not None else H

#     if lags_to_consider is None:
#         lags_to_consider = {}
#     if transformations is None:
#         transformations = {}

#     remaining_lags = {col: list(range(1, ml + 1)) for col, ml in lags_to_consider.items()}
#     remaining_transforms = {col: list(tl) for col, tl in transformations.items()}
#     remaining_feats = list(candidate_features) if candidate_features is not None else []

#     best_lags = {col: [] for col in lags_to_consider}
#     if starting_lags is not None:
#         for col, lags in starting_lags.items():
#             best_lags[col] = list(lags)
#             remaining_lags[col] = [x for x in remaining_lags.get(col, []) if x not in lags]

#     best_transforms = {col: [] for col in transformations}
#     if starting_transforms is not None:
#         for col, tl in starting_transforms.items():
#             best_transforms[col] = list(tl)
#             remaining_transforms[col] = [t for t in remaining_transforms.get(col, []) if t not in tl]

#     best_features = {
#         "best_lags":       best_lags,
#         "best_exogs":      [],
#         "best_transforms": best_transforms,
#     }

#     if candidate_features is not None:
#         df_work = df.drop(columns=candidate_features)
#         df_orig = df.copy()
#     else:
#         df_work = df.copy()
#         df_orig = df.copy()

#     best_score = [float('inf')] * len(metrics)

#     _cat_exog_candidates = set()
#     if candidate_features is not None and model.cat_variables is not None:
#         _cat_exog_candidates = set(candidate_features) & set(model.cat_variables)

#     def _scores_improved(new_score, ref_score):
#         return all(n < r for n, r in zip(new_score, ref_score))

#     def _validate(m, df_test):
#         # cross_validate returns (metrics_df, cv_df_); metrics_df has columns
#         # ["eval_metric", "score"] — one row per metric.
#         m.cross_validate(
#             df=df_test, target_col=target_col, cv_split=cv_split,
#             test_size=H, metrics=metrics, step_size=_step_size,
#         )
#         result_df = m.cv_summary
#         return result_df['overall_score'].tolist()

#     def _make_candidate_model(lags_dict, transforms_dict, active_exogs=None):
#         m = model.copy()
#         # Use {} (not None) when no lags — ml_mv_forecaster.data_prep iterates m.n_lag
#         m.n_lag = {col: sorted(v) for col, v in lags_dict.items() if v}
#         at = {col: list(v) for col, v in transforms_dict.items() if v}
#         m.lag_transform = at if at else None
#         if m.cat_variables is not None and _cat_exog_candidates:
#             active = set(active_exogs) if active_exogs else set()
#             present = [c for c in m.cat_variables
#                        if c not in _cat_exog_candidates or c in active]
#             m.cat_variables = present if present else None
#         return m

#     # Evaluate starting point if warm-start provided
#     if starting_lags is not None or starting_transforms is not None:
#         m_start = _make_candidate_model(
#             best_features["best_lags"],
#             best_features["best_transforms"],
#             best_features["best_exogs"],
#         )
#         best_score = _validate(m_start, df_work)
#         if verbose:
#             print(f"Baseline score: {best_score}")

#     while True:
#         improvement    = False
#         best_candidate = {'target': None, 'type': None, 'value': None}
#         running_score  = best_score

#         # --- try adding each candidate lag ---
#         for col, lags in remaining_lags.items():
#             for lg in lags:
#                 trial_lags = {c: list(v) for c, v in best_features["best_lags"].items()}
#                 trial_lags[col] = sorted(trial_lags[col] + [lg])
#                 m = _make_candidate_model(trial_lags, best_features["best_transforms"],
#                                           best_features["best_exogs"])
#                 score = _validate(m, df_work)
#                 if _scores_improved(score, running_score):
#                     running_score  = score
#                     best_candidate = {'target': col, 'type': 'lag', 'value': lg}
#                     improvement    = True

#         # --- try adding each candidate exog feature ---
#         for feat in remaining_feats:
#             df_test = df_work.copy()
#             df_test[feat] = df_orig[feat]
#             active_exogs = best_features["best_exogs"] + [feat]
#             m = _make_candidate_model(best_features["best_lags"],
#                                       best_features["best_transforms"], active_exogs)
#             score = _validate(m, df_test)
#             if _scores_improved(score, running_score):
#                 running_score  = score
#                 best_candidate = {'target': None, 'type': 'exog', 'value': feat}
#                 improvement    = True

#         # --- try adding each candidate transform ---
#         for col, tl in remaining_transforms.items():
#             for trans in tl:
#                 trial_trans = {c: list(v) for c, v in best_features["best_transforms"].items()}
#                 trial_trans[col] = trial_trans[col] + [trans]
#                 m = _make_candidate_model(best_features["best_lags"], trial_trans,
#                                           best_features["best_exogs"])
#                 score = _validate(m, df_work)
#                 if _scores_improved(score, running_score):
#                     running_score  = score
#                     best_candidate = {'target': col, 'type': 'transform', 'value': trans}
#                     improvement    = True

#         if improvement:
#             best_score = running_score
#             ctype  = best_candidate['type']
#             cval   = best_candidate['value']
#             ctargt = best_candidate['target']
#             if ctype == 'lag':
#                 best_features["best_lags"][ctargt].append(cval)
#                 best_features["best_lags"][ctargt].sort()
#                 remaining_lags[ctargt].remove(cval)
#             elif ctype == 'exog':
#                 best_features["best_exogs"].append(cval)
#                 remaining_feats.remove(cval)
#                 df_work[cval] = df_orig[cval]
#             elif ctype == 'transform':
#                 best_features["best_transforms"][ctargt].append(cval)
#                 remaining_transforms[ctargt].remove(cval)
#             if verbose:
#                 label = cval.get_name() if ctype == 'transform' else cval
#                 tgt_str = f" [{ctargt}]" if ctargt else ""
#                 print(f"Added {ctype}{tgt_str}: {label} | score: {best_score}")
#         else:
#             break

#     for col in best_features["best_lags"]:
#         best_features["best_lags"][col].sort()
#     best_features["best_transforms"] = {
#         col: [t.get_name() for t in tl]
#         for col, tl in best_features["best_transforms"].items()
#     }
#     return best_features

In [25]:
#| hide

# from peshbeen.datasets import load_admission_calls

# from peshbeen.models import ml_mv_forecaster

# admission_calls = load_admission_calls()
# ## get day of week and month as features from the date index
# admission_calls["day_of_week"] = admission_calls.index.dayofweek
# admission_calls["month"] = admission_calls.index.month
# train = admission_calls[:-30]
# test = admission_calls[-30:]

# cat_variables = ["day_of_week", "month"]
# ml_linear = ml_mv_forecaster(model=LinearRegression(),
#               target_cols=['admissions', "calls"], lags = {"admissions": 7, "calls": 7}, categorical_encoder=ohe,
#                 cat_variables=cat_variables,
#                 trend={"admissions": "linear", "calls": "linear"}, change_points={"admissions": [100], "calls": [130]}, pol_degree={"admissions": 1, "calls": 1})
# # ml_linear.fit(train)
# # forecasts = ml_linear.forecast(H=30, exog=test[cat_variables])

In [26]:
#| hide
# feats_mvf = mv_forward_feature_selection(
#     model=ml_linear,
#     df=train,
#     target_col='admissions',
#     cv_split=3,
#     H=30,
#     metrics=[MAE, RMSE],
#     lags_to_consider={"admissions": 7, "calls": 7},
#     candidate_features=cat_variables,
#     verbose=True
# )

In [27]:
#| hide

# def mv_backward_feature_selection(
#     model: object,
#     df: pd.DataFrame,
#     target_col: str,
#     cv_split: int,
#     H: int,
#     step_size=None,
#     metrics=None,
#     lags_to_consider=None,
#     candidate_features=None,
#     transformations=None,
#     verbose=False,
# ):
#     """
#     Backward stepwise feature selection for ``ml_mv_forecaster``.

#     Starts with all candidate features included and iteratively removes
#     the one whose removal most improves cross-validation score.

#     Parameters
#     ----------
#     model : ml_mv_forecaster
#         Template model — never mutated.
#     df : pd.DataFrame
#         All candidate exog columns must already be present.
#     target_col : str
#         Target variable used to evaluate cross-validation score.
#     cv_split : int
#     H : int
#         Forecast horizon / test size per fold.
#     step_size : int, optional
#         Rolling-window step size (defaults to H).
#     metrics : callable or list of callable
#         One or more metric functions (e.g. `[MAE, RMSE]`). A feature is only removed when its removal improves **all** metrics simultaneously.
#     lags_to_consider : dict, optional
#         ``{col: max_lag}`` — all lags 1..max_lag start as selected.
#     candidate_features : list of str, optional
#         Exogenous columns that start as selected.
#     transformations : dict, optional
#         ``{col: [transform_objects]}`` — all transforms start as selected.
#     verbose : bool, default False

#     Returns
#     -------
#     dict
#         ``{"best_lags": {col: [...]}, "best_exogs": [...],
#            "best_transforms": {col: [name_str, ...]}}``
#     """
#     if metrics is None:
#         raise ValueError("metrics must be provided.")
#     if callable(metrics):
#         metrics = [metrics]

#     _step_size = step_size if step_size is not None else H

#     best_features = {
#         "best_lags": {col: list(range(1, ml + 1))
#                       for col, ml in (lags_to_consider or {}).items()},
#         "best_exogs": list(candidate_features) if candidate_features is not None else [],
#         "best_transforms": {col: list(tl) for col, tl in (transformations or {}).items()},
#     }

#     df_work = df.copy()

#     _cat_exog_candidates = set()
#     if candidate_features is not None and model.cat_variables is not None:
#         _cat_exog_candidates = set(candidate_features) & set(model.cat_variables)

#     def _scores_improved(new_score, ref_score):
#         return all(n < r for n, r in zip(new_score, ref_score))

#     def _validate(m, df_test):
#         m.cross_validate(
#             df=df_test, target_col=target_col, cv_split=cv_split,
#             test_size=H, metrics=metrics, step_size=_step_size,
#         )
#         result_df = m.cv_summary
#         return result_df['overall_score'].tolist()

#     def _make_candidate_model(lags_dict, transforms_dict, active_exogs=None):
#         m = model.copy()
#         m.n_lag = {col: sorted(v) for col, v in lags_dict.items() if v}
#         at = {col: list(v) for col, v in transforms_dict.items() if v}
#         m.lag_transform = at if at else None
#         if m.cat_variables is not None and _cat_exog_candidates:
#             active = set(active_exogs) if active_exogs else set()
#             present = [c for c in m.cat_variables
#                        if c not in _cat_exog_candidates or c in active]
#             m.cat_variables = present if present else None
#         return m

#     # Evaluate baseline (all features included) before starting removal
#     m_baseline = _make_candidate_model(
#         best_features["best_lags"],
#         best_features["best_transforms"],
#         best_features["best_exogs"],
#     )
#     best_score = _validate(m_baseline, df_work)
#     if verbose:
#         print(f"Baseline score (all features): {best_score}")

#     while True:
#         improvement    = False
#         best_candidate = {'target': None, 'type': None, 'value': None}
#         running_score  = best_score

#         # --- try removing each selected lag ---
#         for targ_col, lags in best_features["best_lags"].items():
#             for lg in lags:
#                 trial_lags = {col: list(v) for col, v in best_features["best_lags"].items()}
#                 trial_lags[targ_col] = sorted([x for x in lags if x != lg])
#                 m = _make_candidate_model(trial_lags, best_features["best_transforms"],
#                                           best_features["best_exogs"])
#                 score = _validate(m, df_work)
#                 if _scores_improved(score, running_score):
#                     running_score  = score
#                     best_candidate = {'target': targ_col, 'type': 'lag', 'value': lg}
#                     improvement    = True

#         # --- try removing each selected transform ---
#         for targ_col, tl in best_features["best_transforms"].items():
#             for tr in tl:
#                 trial_trans = {col: list(v) for col, v in best_features["best_transforms"].items()}
#                 trial_trans[targ_col] = [x for x in tl if x is not tr]
#                 m = _make_candidate_model(best_features["best_lags"], trial_trans,
#                                           best_features["best_exogs"])
#                 score = _validate(m, df_work)
#                 if _scores_improved(score, running_score):
#                     running_score  = score
#                     best_candidate = {'target': targ_col, 'type': 'transform', 'value': tr}
#                     improvement    = True

#         # --- try removing each selected exog feature ---
#         for feat in best_features["best_exogs"]:
#             df_test      = df_work.drop(columns=[feat])
#             active_after = [x for x in best_features["best_exogs"] if x != feat]
#             m = _make_candidate_model(best_features["best_lags"],
#                                       best_features["best_transforms"], active_after)
#             score = _validate(m, df_test)
#             if _scores_improved(score, running_score):
#                 running_score  = score
#                 best_candidate = {'target': None, 'type': 'exog', 'value': feat}
#                 improvement    = True

#         if improvement:
#             best_score = running_score
#             ctype  = best_candidate['type']
#             cval   = best_candidate['value']
#             ctargt = best_candidate['target']
#             if ctype == 'lag':
#                 best_features["best_lags"][ctargt].remove(cval)
#             elif ctype == 'exog':
#                 best_features["best_exogs"].remove(cval)
#                 df_work = df_work.drop(columns=[cval])
#             elif ctype == 'transform':
#                 best_features["best_transforms"][ctargt].remove(cval)
#             if verbose:
#                 label = cval.get_name() if ctype == 'transform' else cval
#                 tgt_str = f" [{ctargt}]" if ctargt else ""
#                 print(f"Removed {ctype}{tgt_str}: {label} | score: {best_score}")
#         else:
#             break

#     for col in best_features["best_lags"]:
#         best_features["best_lags"][col].sort()
#     best_features["best_transforms"] = {
#         col: [t.get_name() for t in tl]
#         for col, tl in best_features["best_transforms"].items()
#     }
#     return best_features

In [28]:
#| hide

# feats_mvb = mv_backward_feature_selection(
#     model=ml_linear,
#     df=train,
#     target_col='admissions',
#     cv_split=3,
#     H=30,
#     metrics=[MAE, RMSE],
#     lags_to_consider={"admissions": 7, "calls": 7},
#     candidate_features=cat_variables,
#     verbose=True
# )

In [29]:
#| hide

# from peshbeen.models import var
# var_model = var(target_cols=['admissions', "calls"], lags={'admissions': 7, "calls": 7}, trend={'admissions': "linear", "calls": "linear"},
#                 cat_variables=cat_variables, change_points={'admissions': [100], "calls": [130]},
#                 categorical_encoder=ohe)
# feats_mvf_var = mv_forward_feature_selection(
#     model=var_model,
#     df=train,
#     target_col='admissions',
#     cv_split=3,
#     H=30,
#     metrics=[MAE, RMSE],
#     lags_to_consider={"admissions": 7, "calls": 7},
#     candidate_features=cat_variables,
#     verbose=True
# )
# feats_mvf_var

In [30]:
#| hide

# feats_mvb_var = mv_backward_feature_selection(
#     model=var_model,
#     df=train,
#     target_col='admissions',
#     cv_split=3,
#     H=30,
#     metrics=[MAE, RMSE],
#     lags_to_consider={"admissions": 15, "calls": 15},
#     verbose=True
# )
# feats_mvb_var